# Required Skills Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook uses an LLM to extract the required skills from the AI engineering job postings, saves them to `jobs/3-extracted_skills.jsonl`, and then counts which skills appear most often.

In [13]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

### Load classified jobs from file

In [3]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"
skills_jsonl_path = Path("jobs") / "3-extracted_skills.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_df = pd.read_json(classified_jobs_path, lines=True)

print(f"Loaded {len(jobs_df)} classified AI engineering job postings")

Loaded 11 classified AI engineering job postings


### Define the skill categories

We give the model a small set of categories so that the extracted skills are easier to compare across postings.

In [1]:
skill_categories = [
    "ai-engineering",
    "ml",
    "data",
    "backend",
    "cloud",
    "ops",
    "languages",
    "frontend",
    "other",
]

category_text = "\n".join(f"- {category}" for category in skill_categories)
print(category_text)

- ai-engineering
- ml
- data
- backend
- cloud
- ops
- languages
- frontend
- other


### Define the prompt and output schema

We ask the model to extract concrete skills and return them in a structured format.

In [4]:
client = OpenAI()

instructions = """
You extract required skills from AI engineering job postings.

Return concise normalized skill names like 'python', 'rag', 'sql', 'aws', or 'docker'.
Only include skills that are clearly important for the role.
Assign each skill to one of the provided categories.
""".strip()

skill_schema = {
    "type": "object",
    "properties": {
        "skills": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "category": {"type": "string", "enum": skill_categories},
                },
                "required": ["skill", "category"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["skills"],
    "additionalProperties": False,
}

### Extract the skills for each job posting

We loop over the classified AI engineering jobs and ask the model which skills are required for each role.

In [ ]:
skill_results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]
    job_url = job["job_url"]

    print(f"Extracting skills for job {i}/{len(jobs_df)}: {title}")

    prompt = f"""
Extract the required skills for this AI engineering job posting.

Use only these categories:
{category_text}

Description:
{description}
""".strip()

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_skill_extraction",
                "schema": skill_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)

    extracted_skills = [
        {
            "skill": item["skill"].strip().lower(),
            "category": item["category"],
        }
        for item in result["skills"]
    ]

    skill_results.append(
        {
            "title": title,
            "job_url": job_url,
            "extracted_skills": extracted_skills,
            "extracted_skill_labels": [
                f"{item['skill']} [{item['category']}]" for item in extracted_skills
            ],
        }
    )

Extracting skills for job 1/11: Applied AI Engineer Co-op – Fall 2026 (September Start)
[{'skill': 'python', 'category': 'languages'}, {'skill': 'c#', 'category': 'languages'}, {'skill': 'c++', 'category': 'languages'}, {'skill': 'sql', 'category': 'languages'}, {'skill': 'machine learning', 'category': 'ml'}, {'skill': 'large language models', 'category': 'ml'}, {'skill': 'computer vision', 'category': 'ml'}, {'skill': 'data engineering', 'category': 'data'}, {'skill': 'distributed data systems', 'category': 'data'}, {'skill': 'cloud platforms', 'category': 'cloud'}, {'skill': 'aws', 'category': 'cloud'}, {'skill': 'azure', 'category': 'cloud'}, {'skill': 'ai solutions', 'category': 'ai-engineering'}, {'skill': 'architecture', 'category': 'backend'}]
Extracting skills for job 2/11: Senior ML/AI Engineer
[{'skill': 'machine learning', 'category': 'ml'}, {'skill': 'artificial intelligence', 'category': 'ml'}, {'skill': 'ml product deployment', 'category': 'ai-engineering'}, {'skill': 'l

### Save the extracted skills

We turn the list of model outputs into a dataframe and save it as a JSONL file for the next steps.

In [6]:
skills_by_job = pd.DataFrame(skill_results)
skills_by_job.to_json(
    skills_jsonl_path, orient="records", lines=True, force_ascii=False
)

print(f"Saved extracted skills to: {skills_jsonl_path.resolve()}")

skills_by_job_to_show = skills_by_job[["title", "extracted_skill_labels"]].copy()
display(skills_by_job_to_show)

Saved extracted skills to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/jobs/3-extracted_skills.jsonl


,title,extracted_skill_labels
1,Applied AI Engineer Co-op – Fall 2026 (Septemb...,"[python [languages], c# [languages], c++ [lang..."
2,Senior ML/AI Engineer,"[machine learning [ml], llm [ai-engineering], ..."
3,AI (Artificial Intelligence) Engineer (Remote),"[python [languages], api integration [backend]..."
4,AI Engineer,"[python [languages], llm applications [ai-engi..."
5,Lead AI Engineer (AI Foundations),"[python [languages], go [languages], java [lan..."
6,Senior ML/AI Engineer,"[machine learning [ml], artificial intelligenc..."
7,AI Engineer,"[python [languages], llm [ai-engineering], rag..."
8,AI Engineer,"[llm [ai-engineering], agents [ai-engineering]..."
9,Python AI Engineer,"[python [languages], fastapi [backend], langgr..."
10,AI Engineer Intern,"[python [languages], fastapi [backend], postgr..."


### Count how often each skill appears

Now we switch from per-job extraction to a small data analysis step. We count how many job postings mention each skill.

In [8]:
exploded_skills = skills_by_job.explode("extracted_skills").dropna(
    subset=["extracted_skills"]
)

exploded_skills["skill"] = exploded_skills["extracted_skills"].str["skill"]
exploded_skills["category"] = exploded_skills["extracted_skills"].str["category"]

total_jobs = len(skills_by_job)

category_stats = (
    exploded_skills.groupby("category")
    .size()
    .reset_index(name="count")
    .sort_values(by=["count", "category"], ascending=[False, True])
    .reset_index(drop=True)
)

skill_stats = (
    exploded_skills.groupby(["category", "skill"])
    .size()
    .reset_index(name="count")
    .sort_values(
        by=["count", "category", "skill"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

category_stats["share_of_jobs"] = category_stats["count"].apply(
    lambda count: f"{count}/{total_jobs} ({round(count / total_jobs * 100, 1)}%)"
)

skill_stats["share_of_jobs"] = skill_stats["count"].apply(
    lambda count: f"{count}/{total_jobs} ({round(count / total_jobs * 100, 1)}%)"
)

### Display the most common skills

We first look at the categories, then at the most common skills overall, and finally at the top skills inside each category.

In [14]:
top_10_skills = skill_stats[["category", "skill", "share_of_jobs"]].head(10).copy()
top_10_skills.columns = ["Category", "Skill", "Share of jobs"]

display(Markdown("### Most common skills overall"))
display(top_10_skills)

for category in category_stats["category"]:
    top_skills = skill_stats[skill_stats["category"] == category][
        ["skill", "share_of_jobs"]
    ].head(10)

    if not top_skills.empty:
        top_skills = top_skills.copy()
        top_skills.columns = ["Skill", "Share of jobs"]
        display(Markdown(f"### Top skills in `{category}`"))
        display(top_skills)

### Most common skills overall

,Category,Skill,Share of jobs
0,languages,python,9/11 (81.8%)
1,ai-engineering,llm,6/11 (54.5%)
2,ai-engineering,prompt engineering,5/11 (45.5%)
3,ai-engineering,rag,5/11 (45.5%)
4,cloud,aws,5/11 (45.5%)
5,cloud,azure,4/11 (36.4%)
6,ops,observability,4/11 (36.4%)
7,ai-engineering,inference optimization,3/11 (27.3%)
8,ai-engineering,langchain,3/11 (27.3%)
9,ai-engineering,llamaindex,3/11 (27.3%)


### Top skills in `ai-engineering`

,Skill,Share of jobs
1,llm,6/11 (54.5%)
2,prompt engineering,5/11 (45.5%)
3,rag,5/11 (45.5%)
7,inference optimization,3/11 (27.3%)
8,langchain,3/11 (27.3%)
9,llamaindex,3/11 (27.3%)
12,agentic ai,2/11 (18.2%)
13,agentic workflows,2/11 (18.2%)
14,guardrails,2/11 (18.2%)
15,langgraph,2/11 (18.2%)


### Top skills in `ml`

...

,Skill,Share of jobs
6,observability,4/11 (36.4%)
116,agile,1/11 (9.1%)
117,ci/cd,1/11 (9.1%)
118,continuous quality monitoring,1/11 (9.1%)
119,docker,1/11 (9.1%)
120,drift detection,1/11 (9.1%)
121,evaluation pipelines,1/11 (9.1%)
122,experiment tracking,1/11 (9.1%)
123,governance,1/11 (9.1%)
124,langfuse,1/11 (9.1%)


### Top skills in `frontend`

,Skill,Share of jobs
25,react,2/11 (18.2%)
97,next.js,1/11 (9.1%)


### Top skills in `other`

,Skill,Share of jobs
30,human-computer interaction,2/11 (18.2%)
126,proof of concept,1/11 (9.1%)
